In [1]:
import sys
sys.path.append('../../Simulate/')

from UtilityFunctions import retrieve_iupac

In [2]:
import subprocess
import numpy as np

from typing import Dict

In [3]:
class StreamWGSIM:
    '''
    stream WGSIM output for bisulfite reads generation
    :param str sim_cmd: WGSIM commands for simulation
    :param bool pair_end: pair_end or not
    :rtype None
    '''
    def __init__(self, sim_cmd: list = None, pair_end: bool = True):
        self.sim_cmd  = sim_cmd
        self.pair_end = pair_end


    def __iter__(self):
        wgsim = subprocess.Popen(self.sim_cmd, stdout=subprocess.PIPE, universal_newlines=True)
        sim_iter = iter(wgsim.stdout.readline, b'')

        line  = self.get_line(sim_iter) # line is None when EOF
        while line:
            # collect all variant lines on the contig, after that sim_iter points to read lines
            if line == "Contig Variant Start":
                variant_contig, variant_dict = self.collect_variants(sim_iter)
                yield variant_contig, variant_dict

            # collect read pairs
            for collect_flag, read_pair in self.collect_reads(sim_iter):
                if collect_flag: # {1: collect_reads, 0: swith to collect_vars or EOF}
                    yield False, read_pair
                else:
                    line = "Contig Variant Start" if isinstance(read_pair, list) else None
                    break


    def collect_variants(self, sim_iter):
        '''collect variant lines from stdout'''
        variant_dict = {}
        variant_info = {}

        while True:
            line = self.get_line(sim_iter)
            if line == 'Contig Variant End':
                return variant_info['chrom'], variant_dict

            variant_info = self.process_variant_line(line)
            if variant_info['pos']:
                assert variant_info['pos'] not in variant_dict
                variant_dict[variant_info['pos']] = variant_info


    def collect_reads(self, sim_iter):
        '''collect read lines from stdout'''
        skip_flag = not self.pair_end

        while True:
            line  = self.get_line(sim_iter)
            if not line: # EOF
                yield 0, None
            elif line == "Contig Variant Start": # switch to collect variants
                yield 0, []
            else:
                read1 = self.process_read_lines(sim_iter, line = line)
                read2 = self.process_read_lines(sim_iter, skip = skip_flag)
                yield 1, [read1, read2]


    @staticmethod
    def get_line(sim_iter):
        '''receive lines from console'''
        try:
            line = next(sim_iter).strip()
        except StopIteration:
            print("End of output\n")
            return None
        else:
            return line

    @staticmethod
    def process_variant_line(line: str) -> Dict:
        '''parse variant lines'''
        line_split = line.split('\t')

        try:
            chrom, pos, ref, alt, heter_flag = line_split
        except ValueError:
            return dict(chrom=line_split[0], pos = None)
        else:
            heter = heter_flag == '+'
            indel = int(ref == '-') - int(alt == '-') # 1 for ref=='-', -1 for alt=='-', o.w. 0
            offset= indel * max(len(ref), len(alt))
            if indel:
                iupac  = None
            else:
                iupac  = retrieve_iupac(alt)
                alt    = list(set(iupac) - set(ref))[0]
            return dict(chrom=chrom, pos=int(pos), ref=ref, alt=alt,
                        offset=offset, heter=heter, indel=indel, iupac=iupac)

    @staticmethod
    def process_read_lines(sim_iter, line = None, skip = False):
        '''parse read lines'''
        if skip:
            next(sim_iter)
            next(sim_iter)
            next(sim_iter)
            next(sim_iter)
            return None

        if not line:
            line = next(sim_iter).strip()
        # header, seq, comment process
        read_id, pair, flag_pos, flag_mut, flag_indel, qual, cgr = line.split(' ')
        cgr = np.frombuffer(cgr.encode(), dtype=np.int8)
        seq = np.frombuffer(next(sim_iter).strip().encode(), dtype=np.int8)
        _, start, end, cover_pos, n_sub, n_indel, insert_size, inner_dist, ofs= next(sim_iter).strip().split(':')
        ofs = np.fromstring(ofs, dtype=np.int8, sep = ',')
        ctx = np.frombuffer(next(sim_iter).strip().encode(), np.int8)
        return dict(read_id=read_id, pair=int(pair), qual = int(qual),
                    flag_pos=int(flag_pos), flag_mut=int(flag_mut), flag_indel=int(flag_indel),
                    start=int(start), end=int(end), cover_pos=int(cover_pos),
                    n_sub=int(n_sub), n_indel=int(n_indel),
                    insert_size=int(insert_size), inner_dist=int(inner_dist),
                    cgr=cgr, seq=seq, ofs=ofs, ctx=ctx)

# Test for different situation

In [4]:
sim_command = ['/home/wbguo/iproject/BSReadSim/WGSIM/wgsim', 
               '-1', '100', '-2', '100','-e','0.005','-d','400','-s','25',
               '-r','0.001', '-N','1000',
               '-R','0.15','-X','0.15',
               '-S','2022',
               '-A','0.05','-h','0', '-m', '1','/home/wbguo/iproject/BSReadSim/test/ref/BSB_test.fa']

sim_command_empty_fasta = ['/home/wbguo/iproject/BSReadSim/WGSIM/wgsim', 
               '-1', '100', '-2', '100','-e','0.005','-d','400','-s','25',
               '-r','0.001', '-N','1000',
               '-R','0.15','-X','0.15',
               '-S','-1',
               '-A','0.05','-h','0', '-m', '1', '/home/wbguo/iproject/BSReadSim/test/ref/empty_fasta']

sim_command_nonexist_fasta = ['/home/wbguo/iproject/BSReadSim/WGSIM/wgsim', 
               '-1', '100', '-2', '100','-e','0.005','-d','400','-s','25',
               '-r','0.001', '-N','1000',
               '-R','0.15','-X','0.15',
               '-S','-1',
               '-A','0.05','-h','0', '-m', '1', '/home/wbguo/iproject/BSBolt/bsbolt/External/WGSIM/empty_fasta']


sim_command_no_snp = ['/home/wbguo/iproject/BSReadSim/WGSIM/wgsim', 
               '-1', '100', '-2', '100','-e','0.005','-d','400','-s','25',
               '-r','0.00', '-N','1000',
               '-R','0.15','-X','0.15',
               '-S','-1',
               '-A','0.05','-h','0', '-m', '1', '/home/wbguo/iproject/BSReadSim/test/ref/BSB_test.fa']

sim_command_small_N = ['/home/wbguo/iproject/BSReadSim/WGSIM/wgsim', 
               '-1', '100', '-2', '100','-e','0.005','-d','400','-s','25',
               '-r','0.001', '-N','10',
               '-R','0.15','-X','0.15',
               '-S','-1',
               '-A','0.05','-h','0', '-m', '1', '/home/wbguo/iproject/BSReadSim/test/ref/BSB_test.fa']

In [5]:
i = 0
for variant_contig, sim_data in StreamWGSIM(sim_command_no_snp):
    if variant_contig:
        print(sim_data)
    
    if variant_contig:
        print(variant_contig)
    if not variant_contig:
        [sim_data[0]['read_id'], sim_data[0]['pair']]
        ++i

{}
chr10
{}
chr11
{}
chr12
{}
chr13
{}
chr14
{}
chr15


[wgsim] seed = 1668797944
[wgsim_core] calculating the total length of the reference sequence...
[wgsim_core] 6 contig sequences, total length: 1961600
[wgsim_core] No contig id specified, will generate 1000 reads from all contigs
[wgsim_core] No VCF input, will generate SNP randomly if mutation rate is nonzero
[wgsim_core] Generated 1000 read pairs, with 0 contain SNP, 0 contain INDEL


In [6]:
sim_data[0]

{'read_id': '@chr15:1046:1443:2',
 'pair': 0,
 'qual': 56,
 'flag_pos': 0,
 'flag_mut': 0,
 'flag_indel': 0,
 'start': 1045,
 'end': 1145,
 'cover_pos': 0,
 'n_sub': 0,
 'n_indel': 0,
 'insert_size': 397,
 'inner_dist': 197,
 'cgr': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int8),
 'seq': array([3, 3, 3, 1, 3, 3, 0, 3, 0, 1, 0, 1, 0, 0, 1, 3, 1, 0, 1, 1, 3, 2,
        3, 3, 1, 1, 1, 1, 0, 2, 0, 3, 2, 0, 3, 2, 2, 0, 1, 3, 3, 3, 1, 3,
        1, 2, 2, 3, 3, 3, 3, 0, 2, 2, 2, 0, 0, 2, 3, 3, 0, 2, 3, 0, 2, 0,
        3, 2, 3, 2, 3, 3, 2, 3, 3, 0, 2, 0, 0, 0, 1, 3, 3, 2, 2, 1, 0, 3,
        3, 3, 1, 0, 2, 3, 0, 2, 0, 2, 2, 0], dtype=int8),
 'ofs': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [6]:
import sys
print(sys.getsizeof(sim_data[0]))

640


In [8]:
from pympler import asizeof
print(asizeof.asizeof(sim_data[0]))

2840


# Test the running time and size of core steps

In [10]:
sim_cmd = ['/home/wbguo/iproject/BSReadSim/WGSIM/wgsim', 
           '-1', '100', '-2', '100','-e','0.005','-d','400','-s','25',
           '-r','0.1', '-N','1000',
           '-R','0.15','-X','0.15',
           '-S','2022',
           '-A','0.05','-h','0', '-m', '1', '/home/wbguo/iproject/BSReadSim/test/ref/BSB_test.fa']

In [11]:
" ".join(sim_cmd)

'/home/wbguo/iproject/BSReadSim/WGSIM/wgsim -1 100 -2 100 -e 0.005 -d 400 -s 25 -r 0.1 -N 1000 -R 0.15 -X 0.15 -S 2022 -A 0.05 -h 0 -m 1 /home/wbguo/iproject/BSReadSim/test/ref/BSB_test.fa'

In [12]:
wgsim = subprocess.Popen(sim_cmd, stdout=subprocess.PIPE, universal_newlines=True)
sim_iter = iter(wgsim.stdout.readline, b'')

[wgsim] seed = 2022
[wgsim_core] calculating the total length of the reference sequence...
[wgsim_core] 6 contig sequences, total length: 1961600
[wgsim_core] No contig id specified, will generate 1000 reads from all contigs
[wgsim_core] No VCF input, will generate SNP randomly if mutation rate is nonzero


In [13]:
def get_line(sim_iter):
    try:
        line = next(sim_iter).strip()
    except StopIteration:
        print("End of output\n")
        return None
    else:
        return line

In [14]:
def process_variant_line(line: str) -> Dict:
    line_split = line.split('\t')

    try:
        chrom, pos, ref, alt, heter_flag = line_split
    except ValueError:
        return dict(chrom=line_split[0], pos = None)
    else:
        heter = heter_flag == '+'
        indel = int(ref == '-') - int(alt == '-') # 1 for ref=='-', -1 for alt=='-', o.w. 0
        offset= indel * max(len(ref), len(alt))
        if indel:
            iupac  = None
        else:
            iupac  = retrieve_iupac(alt)
            alt    = list(set(iupac) - set(ref))[0]
        return dict(chrom=chrom, pos=int(pos), ref=ref, alt=alt,
                    offset=offset, heter=heter, indel=indel, iupac=iupac)


def collect_variants(sim_iter):
    variant_dict = {}

    while True:
        line = get_line(sim_iter)
        if line == 'Contig Variant End':
            return variant_info['chrom'], variant_dict

        variant_info = process_variant_line(line)
        if variant_info['pos']:
            assert variant_info['pos'] not in variant_dict
            variant_dict[variant_info['pos']] = variant_info

In [15]:
v = collect_variants(sim_iter)

In [16]:
v

('chr10',
 {28: {'chrom': 'chr10',
   'pos': 28,
   'ref': 'T',
   'alt': 'A',
   'offset': 0,
   'heter': True,
   'indel': 0,
   'iupac': ('A', 'T')},
  35: {'chrom': 'chr10',
   'pos': 35,
   'ref': 'T',
   'alt': 'G',
   'offset': 0,
   'heter': True,
   'indel': 0,
   'iupac': ('G', 'T')},
  45: {'chrom': 'chr10',
   'pos': 45,
   'ref': 'C',
   'alt': 'T',
   'offset': 0,
   'heter': True,
   'indel': 0,
   'iupac': ('C', 'T')},
  47: {'chrom': 'chr10',
   'pos': 47,
   'ref': 'T',
   'alt': 'G',
   'offset': 0,
   'heter': False,
   'indel': 0,
   'iupac': ('G',)},
  91: {'chrom': 'chr10',
   'pos': 91,
   'ref': 'G',
   'alt': 'A',
   'offset': 0,
   'heter': False,
   'indel': 0,
   'iupac': ('A',)},
  93: {'chrom': 'chr10',
   'pos': 93,
   'ref': 'T',
   'alt': 'C',
   'offset': 0,
   'heter': True,
   'indel': 0,
   'iupac': ('C', 'T')},
  108: {'chrom': 'chr10',
   'pos': 108,
   'ref': 'C',
   'alt': 'G',
   'offset': 0,
   'heter': True,
   'indel': 0,
   'iupac': ('G', 

In [17]:
print(asizeof.asizeof(v))

20902184


In [18]:
len(v[1])

42318

In [19]:
while True:
    x = get_line(sim_iter)
    if x[0] == "@":
        break

y = get_line(sim_iter)
z = get_line(sim_iter)
t = get_line(sim_iter)

In [20]:
x

'@chr10:195903:196280:0 0 1 61440 3 56 \x00\x00\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01\x00\x00'

In [21]:
x.split(' ')

['@chr10:195903:196280:0',
 '0',
 '1',
 '61440',
 '3',
 '56',
 '\x00\x00\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01\x00\x00']

In [22]:
y

'\x00\x00\x03\x03\x01\x00\x00\x02\x00\x02\x01\x00\x02\x00\x00\x03\x02\x01\x03\x00\x01\x03\x00\x02\x03\x01\x00\x03\x03\x01\x00\x00\x00\x02\x00\x00\x02\x00\x00\x03\x03\x02\x00\x03\x00\x03\x01\x00\x02\x00\x00\x00\x01\x03\x02\x00\x00\x00\x02\x03\x03\x00\x00\x02\x02\x01\x00\x03\x02\x03\x03\x02\x00\x00\x02\x00\x03\x03\x02\x03\x02\x00\x00\x02\x02\x03\x03\x00\x02\x00\x00\x00\x00\x00\x03\x02\x03\x02\x00\x00'

In [23]:
z

'+:195902:196003:1:6:1:377:179:0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1'

In [24]:
t

'\x00\x00\x00\x00\x07\x00\x00\x0f\x00\x0f\x03\x00\x0b\x00\x00\x00\x0f\x07\x00\x00\x07\x00\x00\x0f\x00\x07\x00\x00\x00\x07\x00\x00\x00\x0f\x00\x00\x0f\x00\x00\x00\x00\x0f\x00\x00\x00\x00\x03\x00\x0b\x00\x00\x00\x03\x00\x0b\x00\x00\x00\x0f\x00\x00\x00\x00\x0f\x0f\x07\x00\x00\x0f\x00\x00\x0f\x00\x00\x0f\x00\x00\x00\x0f\x00\x0f\x00\x00\x0f\x0f\x00\x00\x00\x0f\x00\x00\x00\x00\x00\x00\x0f\x00\x0f\x00\x00'

In [25]:
np.frombuffer(t.encode('utf-8'), np.int8)

array([ 0,  0,  0,  0,  7,  0,  0, 15,  0, 15,  3,  0, 11,  0,  0,  0, 15,
        7,  0,  0,  7,  0,  0, 15,  0,  7,  0,  0,  0,  7,  0,  0,  0, 15,
        0,  0, 15,  0,  0,  0,  0, 15,  0,  0,  0,  0,  3,  0, 11,  0,  0,
        0,  3,  0, 11,  0,  0,  0, 15,  0,  0,  0,  0, 15, 15,  7,  0,  0,
       15,  0,  0, 15,  0,  0, 15,  0,  0,  0, 15,  0, 15,  0,  0, 15, 15,
        0,  0,  0, 15,  0,  0,  0,  0,  0,  0, 15,  0, 15,  0,  0],
      dtype=int8)

In [28]:
#### 12 us & 2968 byte for a read, used the fputc for output
def process_read_name(line_list: list):
    '''parse read lines'''
    # header, seq, comment process
    read_id, pair, flag_pos, flag_mut, flag_indel, qual, cgr = line_list[0].split(' ')
    cgr = np.frombuffer(cgr.encode(), dtype=np.int8)
    seq = np.frombuffer(line_list[1].encode(), dtype=np.int8)
    _, start, end, cover_pos, n_sub, n_indel, insert_size, inner_dist, ofs= line_list[2].split(':')
    ofs = np.fromstring(ofs, dtype=np.int8, sep = ',')
    ctx = np.frombuffer(line_list[3].encode(), np.int8)
    return dict(read_id=read_id, pair=int(pair), qual = int(qual),
                flag_pos=int(flag_pos), flag_mut=int(flag_mut), flag_indel=int(flag_indel),
                start=int(start), end=int(end), cover_pos=int(cover_pos),
                n_sub=int(n_sub), n_indel=int(n_indel),
                insert_size=int(insert_size), inner_dist=int(inner_dist),
                cgr=cgr, seq=seq, ofs=ofs, ctx=ctx)

In [29]:
%timeit process_read_name([x,y,z,t]) 

12.1 µs ± 34.4 ns per loop (mean ± std. dev. of 7 runs, 100000 loops each)


In [30]:
import sys
obj = process_read_name([x,y,z,t])
sys.getsizeof(obj)

640

In [31]:
asizeof.asizeof(obj)

2968

In [32]:
obj

{'read_id': '@chr10:195903:196280:0',
 'pair': 0,
 'qual': 56,
 'flag_pos': 1,
 'flag_mut': 61440,
 'flag_indel': 3,
 'start': 195902,
 'end': 196003,
 'cover_pos': 1,
 'n_sub': 6,
 'n_indel': 1,
 'insert_size': 377,
 'inner_dist': 179,
 'cgr': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0], dtype=int8),
 'seq': array([0, 0, 3, 3, 1, 0, 0, 2, 0, 2, 1, 0, 2, 0, 0, 3, 2, 1, 3, 0, 1, 3,
        0, 2, 3, 1, 0, 3, 3, 1, 0, 0, 0, 2, 0, 0, 2, 0, 0, 3, 3, 2, 0, 3,
        0, 3, 1, 0, 2, 0, 0, 0, 1, 3, 2, 0, 0, 0, 2, 3, 3, 0, 0, 2, 2, 1,
        0, 3, 2, 3, 3, 2, 0, 0, 2, 0, 3, 3, 2, 3, 2, 0, 0, 2, 2, 3, 3, 0,
        2, 0, 0, 0, 0, 0, 3, 2, 3, 2, 0, 0], dtype=int8),
 'ofs': array([0, 0, 0, 0, 0, 0, 0, 0, 0,

In [33]:
chr(obj['qual'])

'8'

In [34]:
''.join(['ACGT'[i] for i in obj['seq']])

'AATTCAAGAGCAGAATGCTACTAGTCATTCAAAGAAGAATTGATATCAGAAACTGAAAGTTAAGGCATGTTGAAGATTGTGAAGGTTAGAAAAATGTGAA'

In [35]:
while True:
    x1 = get_line(sim_iter)
    if x1[0] == "@":
        break

y1 = get_line(sim_iter)
z1 = get_line(sim_iter)
t1 = get_line(sim_iter)

In [36]:
obj1 = process_read_name([x1,y1,z1,t1])

In [37]:
obj1

{'read_id': '@chr10:195903:196280:0',
 'pair': 1,
 'qual': 56,
 'flag_pos': 1,
 'flag_mut': 61440,
 'flag_indel': 3,
 'start': 196182,
 'end': 196279,
 'cover_pos': 1,
 'n_sub': 6,
 'n_indel': 3,
 'insert_size': 377,
 'inner_dist': 179,
 'cgr': array([0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 3, 0, 3, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 3, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int8),
 'seq': array([0, 0, 0, 3, 2, 0, 1, 0, 1, 0, 2, 2, 0, 2, 2, 0, 3, 0, 1, 1, 3, 3,
        0, 3, 1, 3, 1, 1, 0, 0, 0, 0, 3, 1, 2, 2, 3, 2, 3, 3, 1, 0, 3, 2,
        3, 3, 1, 3, 2, 0, 2, 2, 3, 0, 1, 1, 2, 2, 2, 0, 0, 3, 3, 0, 0, 2,
        0, 1, 3, 3, 2, 0, 0, 0, 2, 1, 3, 1, 3, 3, 3, 2, 2, 0, 0, 3, 0, 1,
        3, 2, 3, 2, 3, 2, 2, 0, 0, 3, 3, 3], dtype=int8),
 'ofs': array([ 0,  0,  0,  0,  0,  0,  0

In [38]:
sim_data = [obj, obj1]

In [39]:
sys.getsizeof(sim_data)

72

In [40]:
print(asizeof.asizeof(sim_data))

4808


In [41]:
%timeit obj1['start'] + np.arange(len(obj1['seq']))

3.27 µs ± 15.9 ns per loop (mean ± std. dev. of 7 runs, 100000 loops each)


In [42]:
x = obj1['start'] + np.arange(len(obj1['seq']))

In [43]:
%timeit x + obj1['ofs']

1.39 µs ± 3.69 ns per loop (mean ± std. dev. of 7 runs, 1000000 loops each)


# Speed & memory test result

In [ ]:
#### 25 us & 640 byte for a read, used the %d for output
ascii_idx = np.array([i for i in range(48,58)] + [i for i in range(97, 103)])
ascii_val = np.array([i for i in range(0,16)])
ascii_arr = np.full(127, -1).astype(np.int8)
ascii_arr[ascii_idx] = ascii_val

def process_read_name2(line_list: list, read_len: int):
    read_id, pair, flag_pos, flag_mut, flag_indel, cgr = line_list[0].split(' ')
    cgr = np.bitwise_and(np.frombuffer(cgr.encode(), dtype=np.int8), 0x03)
    seq = np.bitwise_and(np.frombuffer(line_list[1].encode(), dtype=np.int8), 0x03)
    _, start, end, cover_pos, n_sub, n_indel, insrt_len, insrt_len2, ofs= line_list[2].split(':')
    ofs = np.fromstring(ofs, dtype=np.int8, sep = ',')
    ctx = ascii_arr[np.frombuffer(line_list[3].encode(), np.int8)]
    return dict(read_id=read_id, pair=int(pair), flag_pos=int(flag_pos), flag_mut=int(flag_mut), flag_indel=int(flag_indel),
                start=int(start), end=int(end), cover_pos=int(cover_pos), n_sub=int(n_sub), n_indel=int(n_indel), 
                insrt_len=int(insrt_len), insrt_len2=int(insrt_len2),
                cgr=cgr, seq=seq, ofs=ofs, ctx=ctx)

In [ ]:
#### 38 us & 360 byte for a read, used the %d for output
def process_read_name3(line_list: list, read_len: int):
    arr = np.full([4, read_len], np.NaN)
    read_id, pair, num_var, num_indel, start, end, mut = line_list[0].split(' ')
    arr[0] = np.bitwise_and(np.frombuffer(mut.encode(), dtype=np.int8), 0x03)
    arr[1] = np.bitwise_and(np.frombuffer(line_list[1].encode(), dtype=np.int8), 0x03)
    _, n_sub, n_indel, insrt_len, ofs = line_list[2].split(':')
    arr[2] = np.fromstring(ofs, dtype=np.int8, sep = ',')
    arr[3] = ascii_arr[np.frombuffer(line_list[3].encode(), np.int8)]
    return dict(read_id = read_id, pair = int(pair),  start=int(start), end=int(end), 
                num_var = int(num_var), num_indel = int(num_indel), n_sub = int(n_sub), n_indel = int(n_indel),
                arr = arr)

In [ ]:
### if use %d
%timeit np.fromstring(y, dtype=np.int8)                               # 1.56 us, will give 48-51
%timeit np.frombuffer(y.encode(), dtype=np.int8)                      # 0.9  us, will give 48-51
%timeit np.array(list(y), dtype=np.int8)                              # 13.5 us, will give 0-3
%timeit np.frombuffer(y.encode(), dtype=np.int8) - 48                 # 4.25 us, will give 0-3
%timeit np.bitwise_and(np.frombuffer(y.encode(), dtype=np.int8), 0x3) # 4.38 us, will give 0-3
%timeit ascii_arr[np.frombuffer(y.encode(), dtype=np.int8)]           # 4.8  us, will give 0-3